# Priors for Promoted and Relegated Teams

**Competitions:** Premier League, La Liga, Bundesliga, Serie A, Ligue 1, EFL Championship\
**Purpose:** Quantify how much promoted and relegated teams differ from their new league's average, to set an evidence-based shrinkage target for newly-arrived teams (instead of the plain league mean).\
**Data:** Cached match results from football-data.org (2024/25 and 2025/26 seasons), already stored in the pipeline's `data/` folder — no extra API calls.\
**Methods:** Roster set-differencing to detect promoted vs. relegated teams, goals-for/against ratios vs. league average.\
**Author:** [Victoria Friss de Kereki](https://www.linkedin.com/in/victoria-friss-de-kereki/)\
**Related:** `football-league-predictions` repo — feeds `PROMOTED_TEAM_ATTACK_RATIO` / `RELEGATED_TEAM_ATTACK_RATIO` and their defense counterparts in `3_probabilities.py`\
**Live results:** [football-league-simulator.streamlit.app](https://football-league-simulator.streamlit.app/)

---

**Notebook first written:** `20/08/2026`\
**Last updated:** `21/08/2026`

> This notebook is the evidence behind one specific fix in the simulator: newly promoted and relegated teams have zero matches of history at their new level, so the default of shrinking them toward the league mean quietly overstates a promoted team's survival odds and understates a relegated team's promotion push.
>
> It measures, from real completed seasons, how far off the league average these two groups actually are — separately for promoted and relegated, since they move in opposite directions.
>
> The result is a small, evidence-backed correction, not a new model: everything here is a shrinkage *target*, and it fades out as soon as a team has enough of its own current-season data to speak for itself.

## Setup

We reuse the same match-result CSVs the production pipeline already caches in `data/` — no extra API calls needed for this analysis.

In [1]:
import pandas as pd

DATA = "data"

def teams(df):
    """Set of every team that appears in a season's match results."""
    return set(pd.unique(df[["homeTeam", "awayTeam"]].values.ravel("K")))

def team_gf_ga(df, team):
    """Goals for / against / matches played, for one team in one season."""
    home = df[df.homeTeam == team]
    away = df[df.awayTeam == team]
    gf = home.homeGoals.sum() + away.awayGoals.sum()
    ga = home.awayGoals.sum() + away.homeGoals.sum()
    played = len(home) + len(away)
    return gf, ga, played

def league_avg_goals_per_team_per_match(df):
    """League-wide average goals scored (== conceded) per team per match."""
    matches_per_team = len(df) * 2 / len(teams(df))
    total_goals = df.homeGoals.sum() + df.awayGoals.sum()
    return total_goals / len(teams(df)) / matches_per_team


## Telling promoted and relegated teams apart

The pipeline already knows, for free, which teams are "newly arrived": a team with zero matches in the *previous*-season portion of its blended history. What it doesn't know by default is which *direction* they came from — that only matters for the Championship, which is the one league in our set with a tracked tier both above it (Premier League) and below it (League One, untracked).

We resolve it with a simple set operation: a newly-arrived Championship team is **relegated** if it also appears in last season's Premier League roster; otherwise it's **promoted**.

In [2]:
pl_2024 = pd.read_csv(f"{DATA}/past_premierleague_england_2024.csv")   # PL 2024/25 (completed)
champ_2025 = pd.read_csv(f"{DATA}/past_championship_england_2025.csv")  # Championship 2025/26 (completed)

relegated_teams = teams(pl_2024) & teams(champ_2025)
relegated_teams


{'Ipswich Town FC', 'Leicester City FC', 'Southampton FC'}

## Relegated teams: how do they actually do in the Championship?

The football folk wisdom is that relegated teams "bounce back" — they arrive with Premier League squads and budgets, playing a level down. Let's check it against the three teams relegated in 2024/25, across their full 2025/26 Championship campaign.

In [3]:
avg_champ = league_avg_goals_per_team_per_match(champ_2025)

rows = []
for team in relegated_teams:
    gf, ga, played = team_gf_ga(champ_2025, team)
    rows.append({
        "team": team,
        "played": played,
        "goals_for_pm": gf / played,
        "goals_against_pm": ga / played,
        "attack_ratio": (gf / played) / avg_champ,
        "defense_ratio": (ga / played) / avg_champ,
    })

relegated_df = pd.DataFrame(rows).sort_values("team")
print(relegated_df.to_string(index=False))


             team  played  goals_for_pm  goals_against_pm  attack_ratio  defense_ratio
  Ipswich Town FC      46       1.73913          1.021739      1.341684       0.788239
Leicester City FC      46       1.26087          1.478261      0.972721       1.140431
   Southampton FC      48       1.75000          1.187500      1.350069       0.916118


In [4]:
print(f"n = {len(relegated_df)}")
print(f"Mean attack ratio:  {relegated_df.attack_ratio.mean():.3f}")
print(f"Mean defense ratio: {relegated_df.defense_ratio.mean():.3f}")


n = 3
Mean attack ratio:  1.221
Mean defense ratio: 0.948


All three relegated teams scored above the Championship average, and two of the three conceded below it. That matches the folk wisdom directly: relegated teams keep the sharper attack and the tighter defense of a Premier League squad, at least for the season right after the drop.

**Caveat on sample size:** football-data.org's free tier only serves match data for the current season plus two seasons back, so this is the one full relegated cohort we can pull real match-by-match results for right now (Ipswich, Leicester, Southampton, 2025/26). The direction and rough size of the effect line up with the wider historical pattern — relegated teams finishing top-half or in the promotion mix considerably more often than a random Championship team would — but n=3 is a starting point, not a settled estimate. Worth revisiting once another relegated cohort's season completes and is on file.

## Promoted teams: how much weaker are they, really?

Same idea, but for teams promoted *into* a top flight. Here we're not limited to one league — we can pool the promoted cohort across all of the "big 5": Premier League, La Liga, Bundesliga, Serie A, Ligue 1. More leagues compensates for only having one clean season of before/after rosters to work with under the same API limit.

In [5]:
BIG5 = {
    "premierleague_england": "Premier League",
    "laliga_spain": "La Liga",
    "bundesliga_germany": "Bundesliga",
    "seriea_italy": "Serie A",
    "ligue1_france": "Ligue 1",
}

rows = []
for league_key, league_label in BIG5.items():
    df_2024 = pd.read_csv(f"{DATA}/past_{league_key}_2024.csv")
    df_2025 = pd.read_csv(f"{DATA}/past_{league_key}_2025.csv")
    promoted = teams(df_2025) - teams(df_2024)
    avg = league_avg_goals_per_team_per_match(df_2025)

    for team in promoted:
        gf, ga, played = team_gf_ga(df_2025, team)
        if played == 0:
            continue
        rows.append({
            "league": league_label,
            "team": team,
            "played": played,
            "attack_ratio": (gf / played) / avg,
            "defense_ratio": (ga / played) / avg,
        })

promoted_df = pd.DataFrame(rows).sort_values(["league", "team"])
print(promoted_df.to_string(index=False))


        league               team  played  attack_ratio  defense_ratio
    Bundesliga         1. FC Köln      34      0.890909       1.145455
    Bundesliga       Hamburger SV      34      0.727273       0.981818
       La Liga           Elche CF      38      0.957031       1.113281
       La Liga         Levante UD      38      0.917969       1.191406
       La Liga        Real Oviedo      38      0.507812       1.171875
       Ligue 1         FC Lorient      34      0.997887       1.060255
       Ligue 1            FC Metz      34      0.665258       1.579988
       Ligue 1           Paris FC      34      0.977098       1.039466
Premier League         Burnley FC      38      0.727273       1.435407
Premier League    Leeds United FC      38      0.937799       1.071770
Premier League     Sunderland AFC      38      0.803828       0.918660
       Serie A       AC Pisa 1909      38      0.563991       1.540130
       Serie A       US Cremonese      38      0.694143       1.236443
      

In [6]:
print(f"n = {len(promoted_df)}")
print(f"Mean attack ratio:  {promoted_df.attack_ratio.mean():.3f}")
print(f"Mean defense ratio: {promoted_df.defense_ratio.mean():.3f}")


n = 14
Mean attack ratio:  0.812
Mean defense ratio: 1.184


14 promoted teams across 5 leagues, every single one below-average on attack and all but one above-average on goals conceded. No ambiguity in the direction here, and pooling across leagues gives us a real sample size instead of 2-3 teams from one division.

Two outliers worth flagging rather than hiding: AC Pisa (0.56 attack, 1.54 defense) and Burnley (0.73 attack, 1.44 defense) were both relegation-threatened all season, while Sassuolo and Lorient came in almost exactly at league average. The mean is doing real averaging here, not just confirming a foregone conclusion.

## What we actually ship

Both priors get applied the same way in `3_probabilities.py`: instead of shrinking a newly-arrived team toward the plain league mean, we shrink it toward `league_mean * ratio`. The shrinkage weight (`shrink_per_team`) still grows with matches played, so this prior only matters while the team is new — by mid-season, its own results dominate regardless.

```python
PROMOTED_TEAM_ATTACK_RATIO = 0.769
PROMOTED_TEAM_DEFENSE_RATIO = 1.224
RELEGATED_TEAM_ATTACK_RATIO = 1.220
RELEGATED_TEAM_DEFENSE_RATIO = 0.811
```

These are close to, not identical to, the ratios recomputed above — they were set from the same kind of historical analysis, and the small gap is rounding/methodology noise rather than a different conclusion (the relegated attack ratio in particular, 1.220 vs. 1.221 here, is about as close as two independent estimates get). Nothing here is precise to the third decimal place; the point is the direction and rough size of the correction, not the exact constant. Both should get revisited as more relegated/promoted cohorts complete a full season and roll into the cached data.
